In [0]:
%sql
select * from dev.bronze.partsupp limit 10 

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.partsupp")

# Metadata columns (_ingestion_timestamp, _source_file, _pipeline_run_id) are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("ps_partkey", "int"),
    typed("ps_suppkey", "int"),
    typed("ps_availqty", "int"),
    typed("ps_supplycost", "decimal(18,2)"),
    typed("ps_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze partsupp rows read: {bronze_count}")
display(typed_df.drop("_row_seq").limit(10))

In [0]:
# ── Step 2: Primary Key Validation (COMPOSITE KEY) ──
# Rule: the row is identified by the combination (ps_partkey, ps_suppkey).
# The SAME two conditions apply as for single-column keys:
#   1. Neither key component may be NULL -> NULL_PK
#   2. The combination must be unique       -> DUPLICATE_PK (first occurrence kept, later repeats quarantined)

pk_null = F.col("ps_partkey").isNull() | F.col("ps_suppkey").isNull()
pk_window = Window.partitionBy("ps_partkey", "ps_suppkey").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Numeric Range Validation ──
# Rule: ps_availqty is the available quantity for this part/supplier pair — it must be >= 0.
# Rule: ps_supplycost is the supply cost for the pair — it must be >= 0.
# Negative, or unparseable (NULL after casting) values are invalid.
# Failure codes: INVALID_AVAILQTY, INVALID_SUPPLYCOST

flagged = (flagged
    .withColumn("is_invalid_availqty",  F.col("ps_availqty").isNull()  | (F.col("ps_availqty")  < 0))
    .withColumn("is_invalid_supplycost", F.col("ps_supplycost").isNull() | (F.col("ps_supplycost") < 0)))

print(f"INVALID_AVAILQTY failures:   {flagged.filter('is_invalid_availqty').count()}")
print(f"INVALID_SUPPLYCOST failures: {flagged.filter('is_invalid_supplycost').count()}")

In [0]:
# ── Step 4: Referential Integrity Validation ──
# partsupp is a BRIDGE table with TWO parents, both already cleaned in Silver (parent-before-child order).
#   ps_partkey -> silver.part      (the product being supplied)
#   ps_suppkey -> silver.supplier  (the supplier fulfilling it)
# A NULL or unmatched key is unresolvable -> quarantined.

fk_df = flagged

part_parent = (spark.table(f"{CATALOG}.silver.part")
    .select(F.col("p_partkey").alias("_part_pk")).distinct()
    .withColumn("_part_hit", F.lit(1)))
fk_df = (fk_df.join(part_parent, F.col("ps_partkey") == F.col("_part_pk"), "left")
    .withColumn("is_part", F.col("_part_hit").isNull())
    .drop("_part_pk", "_part_hit"))

supplier_parent = (spark.table(f"{CATALOG}.silver.supplier")
    .select(F.col("s_suppkey").alias("_supplier_pk")).distinct()
    .withColumn("_supplier_hit", F.lit(1)))
fk_df = (fk_df.join(supplier_parent, F.col("ps_suppkey") == F.col("_supplier_pk"), "left")
    .withColumn("is_supplier", F.col("_supplier_hit").isNull())
    .drop("_supplier_pk", "_supplier_hit"))

flagged = fk_df

print(f"INVALID_FK_PART failures:     {flagged.filter('is_part').count()}")
print(f"INVALID_FK_SUPPLIER failures:  {flagged.filter('is_supplier').count()}")

In [0]:
# ── Step 5: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK")),
        F.when(F.col("is_invalid_availqty"), F.lit("INVALID_AVAILQTY")),
        F.when(F.col("is_invalid_supplycost"), F.lit("INVALID_SUPPLYCOST")),
        F.when(F.col("is_part"), F.lit("INVALID_FK_PART")),
        F.when(F.col("is_supplier"), F.lit("INVALID_FK_SUPPLIER"))
    ),
)

out_cols = ["ps_partkey", "ps_suppkey", "ps_availqty", "ps_supplycost", "ps_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.partsupp"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.partsupp_quarantine"))

print("Tables written: silver.partsupp, silver.partsupp_quarantine")

In [0]:
# ── Step 6: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.partsupp").count()
silver_total     = spark.table(f"{CATALOG}.silver.partsupp").count()
quarantine_total = spark.table(f"{CATALOG}.silver.partsupp_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 7: Data Quality Report ──
# Every check is listed, even when zero rows fail.
rules = [
    ("NULL_PK", "(ps_partkey, ps_suppkey) — neither key component may be null", "is_null_pk"),
    ("DUPLICATE_PK", "(ps_partkey, ps_suppkey) combination must be unique (first occurrence kept)", "is_duplicate_pk"),
    ("INVALID_AVAILQTY", "ps_availqty must be >= 0", "is_invalid_availqty"),
    ("INVALID_SUPPLYCOST", "ps_supplycost must be >= 0", "is_invalid_supplycost"),
    ("INVALID_FK_PART", "ps_partkey must exist in silver.part", "is_part"),
    ("INVALID_FK_SUPPLIER", "ps_suppkey must exist in silver.supplier", "is_supplier")
]
dq_rows = [("partsupp", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_partsupp"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.partsupp_quarantine").limit(10))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver partsupp ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.partsupp").limit(20))